In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import distance

def fit_sphere_hough(points, radius_range=(0.1, 10.0), radius_step=0.1, min_votes=10):
    """
    Fit a sphere to 3D points using Hough Transform.
    
    Parameters:
    points: ndarray of shape (n, 3) - Input 3D points
    radius_range: tuple - Range of possible radii (min, max)
    radius_step: float - Step size for radius sampling
    min_votes: int - Minimum number of votes required for a valid sphere
    
    Returns:
    center: ndarray - Estimated sphere center (x, y, z)
    radius: float - Estimated sphere radius
    """
    
    # Create accumulator for center points
    min_bounds = np.min(points, axis=0)
    max_bounds = np.max(points, axis=0)
    
    # Discretize the search space
    grid_size = 0.5  # Grid resolution for center voting
    x_range = np.arange(min_bounds[0], max_bounds[0], grid_size)
    y_range = np.arange(min_bounds[1], max_bounds[1], grid_size)
    z_range = np.arange(min_bounds[2], max_bounds[2], grid_size)
    r_range = np.arange(radius_range[0], radius_range[1], radius_step)
    
    # Initialize accumulator
    accumulator = np.zeros((len(x_range), len(y_range), len(z_range), len(r_range)))
    
    # Hough Transform voting
    for point in points:
        for i, x in enumerate(x_range):
            for j, y in enumerate(y_range):
                for k, z in enumerate(z_range):
                    center = np.array([x, y, z])
                    dist = distance.euclidean(point, center)
                    
                    # Find closest radius bin
                    r_idx = np.argmin(np.abs(r_range - dist))
                    
                    # Vote for this center-radius combination
                    accumulator[i, j, k, r_idx] += 1
    
    # Find maximum votes
    max_idx = np.unravel_index(np.argmax(accumulator), accumulator.shape)
    center_idx = max_idx[:3]
    radius_idx = max_idx[3]
    
    # Check if we have enough votes
    if accumulator[max_idx] < min_votes:
        return None, None
    
    # Get center and radius
    center = np.array([
        x_range[center_idx[0]],
        y_range[center_idx[1]],
        z_range[center_idx[2]]
    ])
    radius = r_range[radius_idx]
    
    return center, radius

# Example usage
if __name__ == "__main__":
    # Generate sample points on a sphere with some noise
    np.random.seed(0)
    true_center = np.array([0, 0, 0])
    true_radius = 5.0
    n_points = 100
    
    theta = np.random.uniform(0, 2*np.pi, n_points)
    phi = np.random.uniform(0, np.pi, n_points)
    noise = np.random.normal(0, 0.1, (n_points, 3))
    
    points = np.zeros((n_points, 3))
    points[:, 0] = true_center[0] + true_radius * np.sin(phi) * np.cos(theta)
    points[:, 1] = true_center[1] + true_radius * np.sin(phi) * np.sin(theta)
    points[:, 2] = true_center[2] + true_radius * np.cos(phi)
    points += noise
    
    
    # Fit sphere
    estimated_center, estimated_radius = fit_sphere_hough(points)
    
    if estimated_center is not None:
        print(f"Estimated center: {estimated_center}")
        print(f"Estimated radius: {estimated_radius}")
    else:
        print("Failed to fit sphere - insufficient votes")

KeyboardInterrupt: 